[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/main_notebook.ipynb)

# ResNet-18: aumentos básicos frente a AugMix

Dos modelos CNN con **los mismos pesos iniciales ImageNet**, entrenados con imágenes bilineales **128 × 128 (×4)**:

| Experimento | Entrenamiento | Archivo para Kaggle |
|---|---|---|
| Baseline | Crop + flip, entropía cruzada | `submission_baseline.csv` |
| AugMix | Mismos crop + flip, dos vistas AugMix y consistencia JSD | `submission_augmix.csv` |

Selecciona **Entorno de ejecución → Cambiar tipo de entorno → GPU A100** y ejecuta las celdas en orden. No necesitas clonar el repositorio ni ejecutar los scripts `.py`. La celda de actualización es opcional.

Configuración inicial: hasta **30 épocas por modelo**, parada temprana tras 8 épocas sin mejora (a partir de la época 15) y presupuesto aproximado de **2,5 h por modelo y ejecución**. El tiempo se estima después de la primera época; no es una duración garantizada. La inferencia y la preparación quedan fuera de ese presupuesto. AugMix procesa tres vistas por imagen y cuesta más que el baseline.

Se conserva el mejor checkpoint según **validación limpia**, con desempate por pérdida. Los checkpoints se guardan en tu Drive y se reanudan al ejecutar otra vez. Train y validation son CIFAR-10; el test es sintético: **la precisión de validación no sustituye la puntuación de Kaggle**. No se consultan etiquetas del test.

## Actualizar el notebook desde GitHub (opcional)
La siguiente celda muestra un enlace a la revisión más reciente. Ábrelo para cargar el nuevo código: una copia antigua de Drive no se actualiza sola. Los checkpoints se recuperan desde Drive, pero los cambios y resultados de esta pestaña no se trasladan a la nueva.

In [ ]:
import json
import re
import time
import urllib.request
from IPython.display import HTML, display

try:
    request = urllib.request.Request(
        'https://api.github.com/repos/rivas-vega/A3_Recv/commits/main?t=' + str(time.time_ns()),
        headers={'Accept': 'application/vnd.github+json', 'User-Agent': 'A3-Recv-Colab'})
    with urllib.request.urlopen(request, timeout=20) as response:
        latest_sha = json.load(response)['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', latest_sha)
    latest_url = ('https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/'
                  + latest_sha + '/main_notebook.ipynb')
    display(HTML(f'<a href="{latest_url}" target="_blank" rel="noopener noreferrer">'
                 f'Abrir última versión en Colab ({latest_sha[:7]})</a>'))
except Exception as exc:
    print('No se pudo consultar GitHub. Puedes continuar con esta versión:', exc)

## 1. Configuración, autenticación y datos

Mantén `USE_DRIVE=True` para conservar los checkpoints si Colab se desconecta. Para empezar un experimento independiente cambia `RUN_NAME`. Si se modifica una configuración que afecta al entrenamiento, se exige un nombre nuevo para evitar mezclar resultados.

Para Kaggle puedes guardar `KAGGLE_API_TOKEN` en **Secretos** de Colab y permitir acceso a este notebook. Si falta, `kagglehub.login()` pedirá el token de forma interactiva. Se obtiene en [Kaggle → Settings → API → Generate New Token](https://www.kaggle.com/settings/api). Debes haber aceptado las reglas con esa misma cuenta. El login del navegador no autentica el runtime.

In [ ]:
%pip -q install kagglehub

In [ ]:
from pathlib import Path
import csv
import gc
import hashlib
import json
import math
import os
import random
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms as T
from torchvision.transforms import InterpolationMode
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError('Selecciona una GPU en Colab antes de continuar.')
DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

USE_DRIVE = True
RUN_NAME = 'resnet18_augmix_128_v1'
HOURS_PER_MODEL = 2.5  # presupuesto POR ejecución; se puede ampliar al reanudar
NUM_WORKERS = min(4, os.cpu_count() or 1)
EVAL_BATCH_SIZE = 256
RESUME = True

CFG = dict(
    code_version='resnet18_augmix_v1', seed=42, image_size=128,
    weights='ResNet18_Weights.IMAGENET1K_V1', batch_size=128,
    max_epochs=30, min_epochs=15, patience=8, warmup_epochs=2,
    backbone_lr=3e-4, head_lr=1e-3, weight_decay=1e-4,
    augmix_severity=3, augmix_width=3, augmix_depth=-1,
    augmix_alpha=1.0, augmix_all_ops=True, jsd_weight=12.0,
    resize='bilinear_antialias', crop_padding=4,
    torch_version=str(torch.__version__), torchvision_version=str(torchvision.__version__),
    amp_dtype=str(AMP_DTYPE),
)
assert 1 <= CFG['min_epochs'] <= CFG['max_epochs']
assert HOURS_PER_MODEL > 0
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/A3_Recv/runs') / RUN_NAME
else:
    RUN_ROOT = Path('/content/recvis_runs') / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| AMP:', AMP_DTYPE)
print('Checkpoints y resultados:', RUN_ROOT)
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)

In [ ]:
import kagglehub

if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        from google.colab import userdata
        token = userdata.get('KAGGLE_API_TOKEN')
        if token:
            os.environ['KAGGLE_API_TOKEN'] = token
        del token
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass  # login interactivo debajo; nunca se imprime el token
kagglehub.login()
download_root = Path(kagglehub.competition_download('mva-recvis-2026'))
data_files = {}
for filename in ('train.npz', 'validation.npz', 'test.npz', 'sample_submission.csv'):
    matches = list(download_root.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Esperaba un {filename}; encontrados: {matches}')
    data_files[filename] = matches[0]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_labeled(path):
    with np.load(path, allow_pickle=False) as archive:
        images, labels = archive['images'], archive['labels']
    assert images.dtype == np.uint8 and images.shape[1:] == (32, 32, 3)
    assert np.issubdtype(labels.dtype, np.integer)
    labels = labels.reshape(-1).astype(np.int64)
    assert len(images) == len(labels) and len(labels) > 0
    assert ((labels >= 0) & (labels < 10)).all()
    return images, labels

train_images, train_labels = load_labeled(data_files['train.npz'])
val_images, val_labels = load_labeled(data_files['validation.npz'])
CFG['data_sha256'] = {name: sha256_file(path) for name, path in data_files.items()}
config_path = RUN_ROOT / 'config.json'
if config_path.exists():
    if json.loads(config_path.read_text()) != CFG:
        raise ValueError('Configuración/datos distintos: cambia RUN_NAME para un experimento nuevo.')
else:
    config_path.write_text(json.dumps(CFG, indent=2))
print('Train:', train_images.shape, '| Validación:', val_images.shape)

## 2. Transformaciones y función de pérdida

El crop y el flip se aplican a 32 × 32. Las vistas AugMix también se generan a esa resolución, y **todas** se amplían una sola vez con bilinear a 128 × 128 antes de normalizar con las estadísticas de ImageNet. Validación y test solo reciben ese escalado y normalización.

Para cada imagen se comparte la misma geometría básica entre las tres vistas. Las semillas por imagen y época permiten reproducir los aumentos al reanudar y usar los mismos crop/flip en ambos experimentos. Las operaciones GPU rápidas pueden introducir pequeñas variaciones numéricas.

Baseline: entropía cruzada. AugMix: entropía cruzada sobre la vista básica + **12 × JSD** entre las tres predicciones. La pérdida JSD se calcula en float32. Se usa `torchvision.transforms.AugMix` con `all_ops=True`, incluyendo operaciones de color/contraste, severidad 3 y tres ramas. Referencias: [implementación de torchvision](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.AugMix.html) y [AugMix original](https://github.com/google-research/augmix).

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
basic_transform = T.Compose([
    T.RandomCrop(32, padding=CFG['crop_padding'], padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])
preprocess = T.Compose([
    T.Resize((CFG['image_size'], CFG['image_size']),
             interpolation=InterpolationMode.BILINEAR, antialias=True),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
augmix_transform = T.AugMix(
    severity=CFG['augmix_severity'], mixture_width=CFG['augmix_width'],
    chain_depth=CFG['augmix_depth'], alpha=CFG['augmix_alpha'],
    all_ops=CFG['augmix_all_ops'], interpolation=InterpolationMode.BILINEAR,
)

class TrainDataset(Dataset):
    def __init__(self, images, labels, method, epoch):
        assert method in ('baseline', 'augmix')
        self.images, self.labels = images, labels
        self.method, self.epoch = method, epoch

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = Image.fromarray(self.images[index])
        # CPU RNG aislado por imagen; no toca CUDA ni depende del número de workers.
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(
                CFG['seed'] + self.epoch * len(self.images) + index)
            image = basic_transform(image)
            clean = preprocess(image)
            if self.method == 'augmix':
                views = (clean, preprocess(augmix_transform(image)),
                         preprocess(augmix_transform(image)))
            else:
                views = clean
        return views, int(self.labels[index])

class EvalDataset(Dataset):
    def __init__(self, images, targets):
        assert len(images) == len(targets)
        self.images, self.targets = images, targets

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return preprocess(Image.fromarray(self.images[index])), int(self.targets[index])

def make_loader(dataset, training=False, epoch=0):
    generator = torch.Generator().manual_seed(CFG['seed'] + epoch)
    return DataLoader(
        dataset, batch_size=CFG['batch_size'] if training else EVAL_BATCH_SIZE,
        shuffle=training, drop_last=False, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', generator=generator,
        persistent_workers=False,
    )

def js_divergence(logits):
    log_prob = torch.stack([F.log_softmax(x.float(), dim=1) for x in logits])
    prob = log_prob.exp()
    log_mean = torch.logsumexp(log_prob, dim=0) - math.log(len(logits))
    return (prob * (log_prob - log_mean.unsqueeze(0))).sum(dim=-1).mean()

def classification_loss(logits, targets, method):
    if method == 'baseline':
        clean_logits = logits
        ce = F.cross_entropy(clean_logits.float(), targets)
        jsd = ce.new_zeros(())
    elif method == 'augmix':
        clean_logits, augmented1, augmented2 = logits.split(len(targets))
        ce = F.cross_entropy(clean_logits.float(), targets)
        jsd = js_divergence((clean_logits, augmented1, augmented2))
    else:
        raise ValueError(method)
    return ce + CFG['jsd_weight'] * jsd, ce, jsd, clean_logits

val_loader = make_loader(EvalDataset(val_images, val_labels))

### Muestra de entrenamiento (8 imágenes aleatorias)
Solo comprueba que los aumentos conservan el contenido. No se muestra todo el dataset. Esta celda no cambia las semillas del entrenamiento.

In [ ]:
indices = np.random.default_rng(CFG['seed']).choice(len(train_images), 8, replace=False)
preview_set = TrainDataset(train_images, train_labels, 'augmix', epoch=1)
fig, axes = plt.subplots(8, 3, figsize=(7, 16))
mean_t = torch.tensor(MEAN)[:, None, None]
std_t = torch.tensor(STD)[:, None, None]
for row, index in enumerate(indices):
    views, _ = preview_set[int(index)]
    for col, (view, title) in enumerate(zip(views, ('Básica', 'AugMix 1', 'AugMix 2'))):
        axes[row, col].imshow((view * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(title)
plt.tight_layout()
plt.show()

## 3. Entrenamiento reanudable

Ambos modelos ajustan **todas** las capas de una ResNet-18 ImageNet, con el mismo estado inicial y orden de imágenes. AdamW usa LR `3e-4` para el backbone y `1e-3` para la nueva cabeza de 10 clases, dos épocas de calentamiento y descenso coseno. No se utiliza el test para entrenar, seleccionar checkpoints ni ajustar estadísticas de BatchNorm.

Se guarda `last.pt` al terminar cada época (modelo, mejor estado, optimizador, AMP, historial y RNG). Si Colab se interrumpe a mitad, se repite esa época. La tasa de aprendizaje se reconstruye a partir del número de época. Los experimentos ya terminados se cargan sin repetir el entrenamiento. Una parada por tiempo es reanudable; las paradas por paciencia o máximo de épocas son definitivas para ese `RUN_NAME`.

Los dos modelos tienen el mismo máximo y criterio de parada, pero pueden acabar en épocas distintas. La tabla final muestra las épocas, el tiempo y el checkpoint seleccionado de cada uno. El presupuesto temporal se comprueba entre épocas y puede excederse si cambia la velocidad de Colab.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def atomic_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)

def write_history(history, path):
    if not history:
        return
    with open(path, 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)

def build_model(pretrained=False):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model

def lr_factor(epoch):
    warmup, maximum = CFG['warmup_epochs'], CFG['max_epochs']
    if epoch <= warmup:
        return epoch / max(1, warmup)
    progress = (epoch - warmup) / max(1, maximum - warmup)
    return 0.01 + 0.99 * (1 + math.cos(math.pi * progress)) / 2

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    loss_sum = correct = count = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        loss_sum += F.cross_entropy(logits.float(), labels, reduction='sum').item()
        correct += (logits.argmax(1) == labels).sum().item()
        count += len(labels)
    return loss_sum / count, correct / count

def train_epoch(model, loader, optimizer, scaler, method):
    model.train()
    sums = np.zeros(3, dtype=np.float64)
    correct = count = 0
    for views, labels in loader:
        # Orden: [vista básica del batch, AugMix 1 del batch, AugMix 2 del batch].
        images = torch.cat(views, dim=0) if method == 'augmix' else views
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        loss, ce, jsd, clean_logits = classification_loss(logits, labels, method)
        if not torch.isfinite(loss):
            raise FloatingPointError('Pérdida no finita; se conserva el último checkpoint completo.')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        size = len(labels)
        sums += np.array([loss.item(), ce.item(), jsd.item()]) * size
        correct += (clean_logits.argmax(1) == labels).sum().item()
        count += size
    return dict(train_loss=sums[0] / count, train_ce=sums[1] / count,
                train_jsd=sums[2] / count, train_accuracy=correct / count)

def train_model(method):
    assert method in ('baseline', 'augmix')
    seed_everything(CFG['seed'])
    run_dir = RUN_ROOT / method
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / 'last.pt'
    if checkpoint_path.exists() and not RESUME:
        raise ValueError('Hay checkpoints: usa RESUME=True o cambia RUN_NAME.')
    model = build_model().to(DEVICE, memory_format=torch.channels_last)
    model.load_state_dict(INITIAL_STATE)
    optimizer = torch.optim.AdamW([
        {'params': [p for name, p in model.named_parameters() if not name.startswith('fc.')],
         'lr': CFG['backbone_lr']},
        {'params': model.fc.parameters(), 'lr': CFG['head_lr']},
    ], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16)
    history, best_state = [], None
    best_accuracy, best_loss, best_epoch, bad_epochs = -1.0, float('inf'), 0, 0
    start_epoch, finished = 1, False
    stop_reason = 'max_epochs'
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        if checkpoint['config'] != CFG or checkpoint['method'] != method:
            raise ValueError('Checkpoint incompatible; usa otro RUN_NAME.')
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scaler.load_state_dict(checkpoint['scaler'])
        history, best_state = checkpoint['history'], checkpoint['best_model']
        best_accuracy, best_loss = checkpoint['best_accuracy'], checkpoint['best_loss']
        best_epoch, bad_epochs = checkpoint['best_epoch'], checkpoint['bad_epochs']
        start_epoch, finished = checkpoint['epoch'] + 1, checkpoint['finished']
        stop_reason = checkpoint['stop_reason']
        torch.set_rng_state(checkpoint['torch_rng'])
        if DEVICE.type == 'cuda' and checkpoint['cuda_rng'] is not None:
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        print(f'{method}: recuperada época {start_epoch - 1}; mejor validación {best_accuracy:.2%}.')
        del checkpoint

    started = time.monotonic()
    previous_seconds = history[-1]['epoch_seconds'] if history else 0
    if not finished:
        for epoch in range(start_epoch, CFG['max_epochs'] + 1):
            elapsed = time.monotonic() - started
            # Siempre permite al menos una época por ejecución al reanudar.
            if epoch > start_epoch and elapsed + 1.15 * previous_seconds > HOURS_PER_MODEL * 3600:
                stop_reason = 'time_budget'
                print(f'{method}: pausa por presupuesto de tiempo; vuelve a ejecutar para continuar.')
                break
            epoch_started = time.monotonic()
            for group, base_lr in zip(optimizer.param_groups, (CFG['backbone_lr'], CFG['head_lr'])):
                group['lr'] = base_lr * lr_factor(epoch)
            loader = make_loader(TrainDataset(train_images, train_labels, method, epoch),
                                 training=True, epoch=epoch)
            train_metrics = train_epoch(model, loader, optimizer, scaler, method)
            val_loss, val_accuracy = evaluate(model, val_loader)
            improved = (val_accuracy > best_accuracy or
                        (val_accuracy == best_accuracy and val_loss < best_loss))
            if improved:
                best_accuracy, best_loss, best_epoch = val_accuracy, val_loss, epoch
                best_state, bad_epochs = cpu_state(model), 0
            else:
                bad_epochs += 1
            history.append(dict(epoch=epoch, **{k: float(v) for k, v in train_metrics.items()},
                                val_loss=val_loss, val_accuracy=val_accuracy,
                                backbone_lr=optimizer.param_groups[0]['lr'],
                                epoch_seconds=time.monotonic() - epoch_started))
            finished = epoch == CFG['max_epochs'] or (epoch >= CFG['min_epochs'] and bad_epochs >= CFG['patience'])
            stop_reason = 'early_stopping' if finished and epoch < CFG['max_epochs'] else ('max_epochs' if finished else 'running')
            atomic_save(dict(
                config=CFG, method=method, epoch=epoch, model=cpu_state(model),
                optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                best_model=best_state, best_accuracy=best_accuracy, best_loss=best_loss,
                best_epoch=best_epoch, bad_epochs=bad_epochs, history=history,
                finished=finished, stop_reason=stop_reason,
                torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None,
            ), checkpoint_path)
            previous_seconds = time.monotonic() - epoch_started
            # Historial CSV incluye guardado; checkpoint conserva duración previa al guardado.
            history[-1]['epoch_seconds'] = previous_seconds
            write_history(history, run_dir / 'history.csv')
            print(f"{method} {epoch:02d}/{CFG['max_epochs']} | train {train_metrics['train_accuracy']:.2%}"
                  f' | val {val_accuracy:.2%} | {previous_seconds / 60:.1f} min'
                  + (' | mejor' if improved else ''), flush=True)
            if epoch == start_epoch:
                remaining = (CFG['max_epochs'] - epoch) * previous_seconds / 3600
                print(f'Estimación para las épocas restantes de {method}: {remaining:.2f} h; sujeta a la velocidad de Colab.')
            del loader
            if finished:
                break
    if best_state is None:
        raise RuntimeError('No hay ningún checkpoint válido; no se generará un CSV sin entrenar.')
    model.load_state_dict(best_state)
    model.eval()
    # Reconstruye también el CSV si hubo una desconexión justo después de guardar last.pt.
    write_history(history, run_dir / 'history.csv')
    atomic_save({'model': best_state, 'config': CFG, 'method': method,
                 'epoch': best_epoch, 'val_accuracy': best_accuracy}, run_dir / 'best.pt')
    summary = dict(method=method, epochs_trained=len(history), best_epoch=best_epoch,
                   val_accuracy=best_accuracy, val_loss=best_loss,
                   training_hours=sum(row['epoch_seconds'] for row in history) / 3600,
                   stop_reason=stop_reason, completed=finished)
    (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    print(f'{method}: se usará época {best_epoch}, validación {best_accuracy:.2%}.')
    del optimizer, scaler, best_state
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return model, summary

In [ ]:
seed_everything(CFG['seed'])
initial_path = RUN_ROOT / 'initial_state.pt'
if initial_path.exists():
    initial_checkpoint = torch.load(initial_path, map_location='cpu', weights_only=True)
    if initial_checkpoint['config'] != CFG:
        raise ValueError('Inicialización incompatible: cambia RUN_NAME.')
    INITIAL_STATE = initial_checkpoint['model']
    del initial_checkpoint
else:
    initial_model = build_model(pretrained=True)
    INITIAL_STATE = cpu_state(initial_model)
    atomic_save({'model': INITIAL_STATE, 'config': CFG}, initial_path)
    del initial_model
print('Inicialización ImageNet compartida lista. Se entrenan todas las capas.')

## 4. Inferencia y los dos CSV

Las predicciones se calculan con el **mejor checkpoint de cada modelo**. Test recibe exactamente la misma transformación bilineal 128 × 128 que validación. No hay AugMix, TTA ni ajuste de estadísticas durante inferencia.

El exportador alinea las predicciones por `id` con `sample_submission.csv`, incluso si el orden difiere del NPZ, y verifica duplicados, longitud y etiquetas enteras 0–9. Primero se completa baseline y se escribe su CSV; después AugMix. Así se conserva el primero si se interrumpe la segunda ejecución.

In [ ]:
def write_submission(ids, predictions, template_path, output_path):
    ids = [int(value) for value in ids]
    raw_predictions = np.asarray(predictions)
    if not np.issubdtype(raw_predictions.dtype, np.integer):
        raise ValueError('Las etiquetas predichas deben ser enteras.')
    predictions = [int(value) for value in raw_predictions]
    if len(ids) != len(predictions) or len(set(ids)) != len(ids):
        raise ValueError('IDs repetidos o longitud incorrecta.')
    if not predictions or not all(0 <= value <= 9 for value in predictions):
        raise ValueError('Predicciones fuera de [0,9] o vacías.')
    with open(template_path, newline='') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ['id', 'label']:
            raise ValueError('La plantilla debe tener columnas id,label.')
        ordered_ids = [int(row['id']) for row in reader]
    if len(ordered_ids) != len(ids) or len(set(ordered_ids)) != len(ordered_ids) or set(ordered_ids) != set(ids):
        raise ValueError('Los IDs del test y la plantilla no coinciden.')
    lookup = dict(zip(ids, predictions))
    output_path = Path(output_path)
    temporary = output_path.with_name(output_path.name + '.tmp')
    with open(temporary, 'w', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(['id', 'label'])
        writer.writerows((identifier, lookup[identifier]) for identifier in ordered_ids)
    os.replace(temporary, output_path)
    return len(ordered_ids)

@torch.inference_mode()
def make_submission(model, method):
    with np.load(data_files['test.npz'], allow_pickle=False) as archive:
        test_images, test_ids = archive['images'], archive['ids']
    assert test_images.dtype == np.uint8 and test_images.shape[1:] == (32, 32, 3)
    assert test_ids.ndim == 1 and np.issubdtype(test_ids.dtype, np.integer)
    test_loader = make_loader(EvalDataset(test_images, test_ids))
    model.eval()
    ids, predictions = [], []
    for images, batch_ids in test_loader:
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        predictions.extend(logits.argmax(1).cpu().tolist())
        ids.extend(batch_ids.tolist())
    output_path = RUN_ROOT / f'submission_{method}.csv'
    count = write_submission(ids, predictions, data_files['sample_submission.csv'], output_path)
    print(f'{output_path.name}: {count:,} predicciones verificadas. Guardado en {output_path}')
    return output_path

In [ ]:
baseline_model, baseline_summary = train_model('baseline')
baseline_csv = make_submission(baseline_model, 'baseline')
del baseline_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
augmix_model, augmix_summary = train_model('augmix')
augmix_csv = make_submission(augmix_model, 'augmix')
del augmix_model
gc.collect()
torch.cuda.empty_cache()

## 5. Comparación y descarga

La tabla se rellena con resultados de las ejecuciones, nunca con cifras de ejemplo. `kaggle_accuracy` permanece vacío hasta que subas **cada CSV por separado** y copies su puntuación pública. El desacuerdo entre modelos no indica cuál acierta. Una mayor precisión en CIFAR-10 no garantiza mejorar sobre el test sintético.

Si un modelo ha parado por tiempo, su resultado es provisional; puedes reanudar su celda de entrenamiento y volver a generar la comparación. Para comparar a igual número de épocas, consulta también las curvas en el intervalo común. No se envían submissions automáticamente.

In [ ]:
# Tras subir los archivos, pega aquí las puntuaciones de Kaggle como fracciones (p. ej. 0.90).
KAGGLE_SCORES = {'baseline': None, 'augmix': None}
rows = []
for method in ('baseline', 'augmix'):
    summary = json.loads((RUN_ROOT / method / 'summary.json').read_text())
    summary['kaggle_accuracy'] = KAGGLE_SCORES[method]
    summary['csv'] = f'submission_{method}.csv'
    rows.append(summary)
comparison = pd.DataFrame(rows)
comparison.to_csv(RUN_ROOT / 'comparison.csv', index=False)
display(comparison.style.format({'val_accuracy': '{:.2%}', 'val_loss': '{:.4f}',
                                'training_hours': '{:.2f}', 'kaggle_accuracy': '{:.2%}'}, na_rep='pendiente'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method in ('baseline', 'augmix'):
    history = pd.read_csv(RUN_ROOT / method / 'history.csv')
    axes[0].plot(history.epoch, 100 * history.val_accuracy, label=method)
    axes[1].plot(history.epoch, history.val_loss, label=method)
axes[0].set(xlabel='Época', ylabel='Precisión validación limpia (%)')
axes[1].set(xlabel='Época', ylabel='Entropía cruzada de validación')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RUN_ROOT / 'validation_curves.png', dpi=150)
plt.show()

baseline_predictions = pd.read_csv(RUN_ROOT / 'submission_baseline.csv').set_index('id')
augmix_predictions = pd.read_csv(RUN_ROOT / 'submission_augmix.csv').set_index('id')
augmix_predictions = augmix_predictions.reindex(baseline_predictions.index)
disagreement = float((baseline_predictions.label != augmix_predictions.label).mean())
print(f'Desacuerdo entre predicciones: {disagreement:.2%} (no es accuracy).')
delta = comparison.set_index('method').val_accuracy['augmix'] - comparison.set_index('method').val_accuracy['baseline']
print(f'Diferencia de validación AugMix − baseline: {100 * delta:+.2f} puntos porcentuales.')
if all(KAGGLE_SCORES[m] is not None for m in ('baseline', 'augmix')):
    print(f"Diferencia Kaggle: {100 * (KAGGLE_SCORES['augmix'] - KAGGLE_SCORES['baseline']):+.2f} puntos.")

In [ ]:
# Un ZIP pequeño que contiene los DOS CSV, el resumen y las curvas (sin pesos ni datos).
# Descomprímelo y sube cada submission_*.csv por separado a Kaggle.
bundle = RUN_ROOT / 'recvis_two_submissions.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in ('submission_baseline.csv', 'submission_augmix.csv', 'comparison.csv',
                     'validation_curves.png', 'config.json'):
        archive.write(RUN_ROOT / filename, arcname=filename)
    for method in ('baseline', 'augmix'):
        archive.write(RUN_ROOT / method / 'history.csv', arcname=f'{method}_history.csv')
print('También disponibles en Drive:', RUN_ROOT)
from google.colab import files
files.download(str(bundle))

Para el informe: declara el preentrenamiento ImageNet (`ResNet18_Weights.IMAGENET1K_V1`), el escalado bilineal ×4, los parámetros AugMix/JSD y las épocas/checkpoints que muestra la tabla. Guarda las puntuaciones Kaggle separadas de la validación. Esta prueba compara dos entrenamientos; no demuestra por sí sola que el efecto se mantenga para otras semillas.

El repositorio es público: conserva los datos, pesos y credenciales en Colab/Drive. Antes de guardar una copia del notebook en GitHub, borra sus salidas. Para la entrega del curso, el ZIP de código es distinto del ZIP de predicciones y debe excluir pesos/datos (límite indicado: 8 MB).